# 1. Security Operations Design — SOC Architecture

SC-100 asks: **"Design a security operations strategy that uses the right combination of SIEM, XDR, and SOAR for a given organization."**

## Setup

Make sure you've installed the kernel:
```bash
cd security/sc-100/02-secops-identity-compliance
uv sync
uv run python -m ipykernel install --user --name=sc-100 --display-name="SC-100 (Python)"
```
Then select the **SC-100 (Python)** kernel in VS Code's kernel picker (top-right).

If the kernel doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window").

## Microsoft's Unified SecOps Architecture

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                        UNIFIED SECURITY OPERATIONS                          │
│                                                                              │
│  ┌─────────────────────────┐      ┌──────────────────────────────────────┐  │
│  │     MICROSOFT SENTINEL  │      │        MICROSOFT DEFENDER XDR        │  │
│  │         (SIEM)          │◄────►│           (XDR platform)             │  │
│  │                         │      │                                      │  │
│  │  • Log collection       │      │  • Defender for Endpoint             │  │
│  │  • KQL analytics        │      │  • Defender for Identity             │  │
│  │  • Custom detections    │      │  • Defender for Office 365           │  │
│  │  • Workbooks            │      │  • Defender for Cloud Apps           │  │
│  │  • SOAR playbooks       │      │  • Defender for Cloud                │  │
│  │  • Threat intelligence  │      │  • Attack disruption (automatic)     │  │
│  │  • 3rd party data       │      │  • Cross-domain correlation          │  │
│  └─────────────────────────┘      └──────────────────────────────────────┘  │
│                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────┐ │
│  │                     UNIFIED PORTAL (security.microsoft.com)            │ │
│  │  Single pane of glass: incidents, alerts, hunting, automation          │ │
│  └─────────────────────────────────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────────────────────────────┘
```

### Key architecture decision: When do you need what?

| Capability | Sentinel (SIEM) | Defender XDR | Both (recommended) |
|-----------|----------------|-------------|-------------------|
| Microsoft 365 protection | ❌ | ✅ native | ✅ |
| 3rd party log ingestion | ✅ | ❌ | ✅ |
| Custom KQL detections | ✅ | Limited | ✅ |
| Automatic attack disruption | ❌ | ✅ | ✅ |
| SOAR playbooks | ✅ Logic Apps | Basic | ✅ |
| Multi-cloud (AWS/GCP) | ✅ | Limited | ✅ |
| Compliance/audit logs | ✅ | ❌ | ✅ |
| Cost for M365-only org | Higher | Included with M365 E5 | Depends |

In [ ]:
import json

# ===================================================================
# SIEM vs XDR DECISION MATRIX
# As an architect, you need to recommend the right tooling
# ===================================================================

DECISION_MATRIX = {
    'Sentinel Only': {
        'best_for': 'Organizations with heavy 3rd-party/on-prem infrastructure, no M365 E5',
        'strengths': ['Custom detections', '3rd party connectors', 'Long-term retention', 'SOAR via Logic Apps'],
        'weaknesses': ['No automatic attack disruption', 'Higher cost for M365 data', 'Requires KQL expertise'],
        'cost_model': 'Pay per GB ingested',
    },
    'Defender XDR Only': {
        'best_for': 'M365 E5 orgs with primarily Microsoft workloads, small SOC teams',
        'strengths': ['Zero config for M365', 'Automatic attack disruption', 'Built-in correlation', 'Included in E5'],
        'weaknesses': ['No 3rd party log ingestion', 'Limited custom detections', 'No SOAR'],
        'cost_model': 'Included with M365 E5 license',
    },
    'Sentinel + Defender XDR (Unified)': {
        'best_for': 'Enterprise/hybrid orgs, regulated industries, mature SOC teams',
        'strengths': ['Full visibility', 'Best of both worlds', 'Unified incident queue', 'Bi-directional sync'],
        'weaknesses': ['Higher total cost', 'More complex to manage', 'Requires skilled analysts'],
        'cost_model': 'E5 license + Sentinel per-GB (free for XDR data forwarded)',
    },
}

print('=== SIEM vs XDR Decision Matrix ===\n')
for approach, details in DECISION_MATRIX.items():
    print(f'\n--- {approach} ---')
    print(f'  Best for: {details["best_for"]}')
    print(f'  Strengths: {", ".join(details["strengths"])}')
    print(f'  Weaknesses: {", ".join(details["weaknesses"])}')
    print(f'  Cost model: {details["cost_model"]}')

## Centralized Logging Architecture

```
                         ┌──────────────────┐
                         │  Microsoft        │
                         │  Sentinel         │
                         │  (Log Analytics   │
                         │   Workspace)      │
                         └────────┬─────────┘
                                  │
        ┌─────────────────────────┼─────────────────────────┐
        │                         │                         │
   ┌────▼────┐             ┌──────▼──────┐          ┌───────▼───────┐
   │ Azure   │             │  On-Prem     │          │  3rd Party    │
   │ Sources │             │  Sources     │          │  Sources      │
   ├─────────┤             ├──────────────┤          ├───────────────┤
   │ Entra ID│             │ AD DS logs   │          │ Palo Alto     │
   │ Activity│             │ DNS/DHCP     │          │ CrowdStrike   │
   │ Diag.   │             │ Syslog       │          │ AWS CloudTrail│
   │ NSG Flow│             │ Windows Evts │          │ GCP Audit     │
   │ Key Vault│            │ CEF/Syslog   │          │ Okta          │
   └─────────┘             └──────────────┘          └───────────────┘
                    (via AMA agent / ARC)       (via data connectors)
```

### Key design decisions for log architecture:

| Decision | Options | Guidance |
|----------|---------|----------|
| Workspace topology | Single vs multi-workspace | Single preferred; multi only for sovereignty/RBAC |
| Data tiers | Analytics vs Basic vs Archive | Hot data → Analytics; low-value → Basic (cheaper); compliance → Archive |
| Retention | 30 days (default) to 12 years | Regulatory = long retention; use Archive tier |
| Agent | AMA (Azure Monitor Agent) | Always AMA; legacy MMA is deprecated |
| Multi-cloud | Azure Arc + connectors | Arc for servers; native connectors for AWS/GCP |

In [ ]:
# ===================================================================
# SCENARIO: Design a SOC architecture for Woodgrove Bank
# ===================================================================

SCENARIO = """
COMPANY: Woodgrove Bank
EMPLOYEES: 12,000
INDUSTRY: Financial services (regulated)
INFRASTRUCTURE:
  - Azure: 8 subscriptions across 3 tenants (due to acquisitions)
  - AWS: 2 accounts (legacy workloads from acquired company)
  - On-premises: 500 Windows servers, AD DS forest
  - M365 E5 for all employees
  - Palo Alto firewalls (on-prem and cloud)
  - CrowdStrike on some acquired-company endpoints

SOC TEAM: 8 analysts (4 Tier-1, 3 Tier-2, 1 Tier-3)

CURRENT PROBLEMS:
  - Alert fatigue: 2,000+ alerts/day, mostly false positives
  - Separate consoles for each security tool
  - No automated response to common incidents
  - 3-day average MTTD (mean time to detect)
  - Compliance requires 7-year log retention

GOALS:
  - Unified view of all security incidents
  - Reduce MTTD from 3 days to under 1 hour
  - Automate response to 80% of Tier-1 alerts
  - Meet regulatory retention requirements
  - Maintain CrowdStrike until endpoint migration completes
"""

print(SCENARIO)

In [ ]:
# ===================================================================
# ARCHITECTURE DESIGN: Evaluate different approaches
# ===================================================================

DESIGN_OPTIONS = {
    'Option A: Sentinel + Defender XDR (Unified)': {
        'architecture': [
            'Single Sentinel workspace (primary tenant)',
            'Azure Lighthouse for cross-tenant visibility',
            'Defender XDR for all M365 + endpoint signals',
            'Sentinel connectors: Palo Alto, CrowdStrike, AWS CloudTrail',
            'Data tiers: Security logs → Analytics, audit logs → Basic, compliance → Archive',
        ],
        'pros': ['Unified incident queue', 'XDR auto-disruption for M365 threats',
                 'Custom KQL for 3rd party', 'SOAR for Tier-1 automation'],
        'cons': ['Cost of Sentinel ingestion for 3rd party data',
                 'Cross-tenant setup complexity'],
        'score': 9,
    },
    'Option B: Sentinel Only': {
        'architecture': [
            'Multi-workspace Sentinel (one per tenant)',
            'All data sources send to Sentinel',
            'Custom analytics rules for everything',
        ],
        'pros': ['Full control', 'Single tool to learn'],
        'cons': ['Lose XDR auto-disruption', 'Higher M365 ingestion cost',
                 'Multi-workspace complexity', 'No built-in correlation'],
        'score': 5,
    },
    'Option C: Defender XDR + 3rd party SIEM': {
        'architecture': [
            'Defender XDR for Microsoft stack',
            'Splunk for 3rd party and on-prem logs',
            'Manual correlation between tools',
        ],
        'pros': ['Leverage existing Splunk investment', 'XDR for M365'],
        'cons': ['Two consoles (not unified)', 'No cross-platform correlation',
                 'Splunk licensing cost', 'Harder to automate'],
        'score': 4,
    },
}

print('=== Architecture Design Evaluation ===\n')
for option, details in sorted(DESIGN_OPTIONS.items(), key=lambda x: x[1]['score'], reverse=True):
    print(f'\n{"=" * 60}')
    print(f'{option}  [Score: {details["score"]}/10]')
    print(f'{"=" * 60}')
    print('Architecture:')
    for step in details['architecture']:
        print(f'  • {step}')
    print(f'Pros: {", ".join(details["pros"])}')
    print(f'Cons: {", ".join(details["cons"])}')

print('\n\n>>> RECOMMENDED: Option A — Sentinel + Defender XDR (Unified)')
print('    REASON: Regulated industry needs full visibility + automation.')
print('    M365 E5 already includes Defender XDR — no extra license cost.')
print('    Sentinel handles 3rd party (Palo Alto, CrowdStrike, AWS) + compliance retention.')

## SOAR Automation Design

SOAR (Security Orchestration, Automation, and Response) is critical for reducing analyst workload.

### Automation tiers in Microsoft Sentinel:

| Tier | Mechanism | Use case | Example |
|------|-----------|----------|---------|
| **Automation rules** | Built-in, no-code | Triage, assign, close | Auto-close known false positives |
| **Playbooks (Logic Apps)** | Low-code workflows | Enrichment, response | Look up IP in TI, block in firewall |
| **Custom notebooks** | Code (Jupyter/KQL) | Investigation, hunting | Threat hunting with MSTICPy |

### Common SOAR playbook patterns:

```
PATTERN 1: Phishing auto-response
  Trigger → Defender for Office 365 alert
    → Extract sender, URLs, attachments
    → Check URL reputation (TI)
    → If malicious: purge email from all mailboxes
    → Block sender domain in Exchange transport rule
    → Notify SOC team via Teams

PATTERN 2: Impossible travel
  Trigger → Entra ID Protection risk detection
    → Check if user has VPN (known IP ranges)
    → If not VPN: require MFA re-authentication
    → If MFA fails: disable account, notify manager
    → Create ServiceNow ticket

PATTERN 3: Malware on endpoint
  Trigger → Defender for Endpoint alert
    → Isolate device from network (API call)
    → Collect investigation package
    → Run AV scan
    → Notify user and IT support
    → Assign to Tier-2 analyst
```

In [ ]:
# ===================================================================
# SOAR AUTOMATION DESIGN EXERCISE
# Design which alerts should be automated vs manual
# ===================================================================

ALERT_CATEGORIES = [
    {'alert': 'Phishing email detected',           'volume': 'High (200/day)',  'complexity': 'Low',    'recommended': 'Full auto',    'reason': 'Well-understood, repeatable response (purge + block)'},
    {'alert': 'Impossible travel sign-in',         'volume': 'Medium (30/day)', 'complexity': 'Low',    'recommended': 'Semi-auto',    'reason': 'Auto-enrich with VPN check, escalate unknowns'},
    {'alert': 'Brute force attack (external)',     'volume': 'High (500/day)',  'complexity': 'Low',    'recommended': 'Full auto',    'reason': 'Auto-block IP after threshold, close alert'},
    {'alert': 'Malware detected on endpoint',      'volume': 'Medium (20/day)', 'complexity': 'Medium', 'recommended': 'Semi-auto',    'reason': 'Auto-isolate device, but analyst reviews before remediation'},
    {'alert': 'Suspicious PowerShell execution',    'volume': 'Low (5/day)',     'complexity': 'High',   'recommended': 'Manual + enrich', 'reason': 'Needs context — could be admin activity or attack'},
    {'alert': 'Data exfiltration detected',         'volume': 'Low (2/day)',     'complexity': 'High',   'recommended': 'Manual',       'reason': 'High impact, requires human judgment and investigation'},
    {'alert': 'Privilege escalation attempt',       'volume': 'Low (3/day)',     'complexity': 'High',   'recommended': 'Semi-auto',    'reason': 'Auto-collect forensics, disable account, escalate to Tier-3'},
    {'alert': 'SSL certificate expiring',           'volume': 'Low (1/week)',    'complexity': 'Low',    'recommended': 'Full auto',    'reason': 'Create ticket in ITSM, no security investigation needed'},
]

print('=== SOAR Automation Design ===\n')
print(f'{"Alert":<40} {"Volume":<20} {"Automation":<20} {"Reason"}')
print('─' * 120)
for a in ALERT_CATEGORIES:
    print(f'{a["alert"]:<40} {a["volume"]:<20} {a["recommended"]:<20} {a["reason"]}')

auto_count = sum(1 for a in ALERT_CATEGORIES if 'auto' in a['recommended'].lower())
total = len(ALERT_CATEGORIES)
print(f'\nAutomation coverage: {auto_count}/{total} alert types ({auto_count/total*100:.0f}%) have some automation')
print('Target: Automate response for 80%+ of Tier-1 alert volume')

## MITRE ATT&CK Coverage Analysis

A cybersecurity architect must design detection coverage that maps to the **MITRE ATT&CK** framework.

The goal is not 100% coverage everywhere — it's **prioritized coverage based on your threat landscape**.

### Key tactics for financial services (Woodgrove Bank):

| Tactic | Priority | Microsoft tool | Coverage |
|--------|----------|---------------|----------|
| Initial Access | Critical | Defender for Office 365, Entra ID Protection | Phishing, credential stuffing |
| Execution | High | Defender for Endpoint | PowerShell, script-based attacks |
| Persistence | High | Defender for Identity, Sentinel | Service principal abuse, scheduled tasks |
| Privilege Escalation | Critical | Defender for Identity, PIM alerts | Kerberoasting, token manipulation |
| Credential Access | Critical | Defender for Identity, Entra ID | Password spray, NTLM relay |
| Lateral Movement | Critical | Defender for Identity, Sentinel | Pass-the-hash, RDP abuse |
| Collection | High | Defender for Cloud Apps, DLP | Email collection, data staging |
| Exfiltration | Critical | Defender for Cloud Apps, Sentinel | Large uploads, anomalous transfers |
| Impact | Critical | Defender for Endpoint, Backup alerts | Ransomware encryption, data destruction |

In [ ]:
# ===================================================================
# MITRE ATT&CK COVERAGE GAP ANALYSIS
# ===================================================================

MITRE_COVERAGE = {
    'Initial Access':        {'current': 85, 'target': 95, 'gaps': ['No SMS phishing detection', 'Voice phishing not covered']},
    'Execution':             {'current': 70, 'target': 90, 'gaps': ['Linux execution monitoring limited', 'Container runtime not monitored']},
    'Persistence':           {'current': 60, 'target': 85, 'gaps': ['Service principal monitoring weak', 'Scheduled tasks on legacy servers']},
    'Privilege Escalation':  {'current': 75, 'target': 95, 'gaps': ['Azure RBAC changes not alerted', 'On-prem GPO modifications']},
    'Defense Evasion':       {'current': 50, 'target': 80, 'gaps': ['Log tampering detection missing', 'AV exclusion monitoring']},
    'Credential Access':     {'current': 80, 'target': 95, 'gaps': ['AS-REP roasting', 'Cloud credential dumping']},
    'Discovery':             {'current': 40, 'target': 70, 'gaps': ['Network scanning detection', 'Cloud resource enumeration']},
    'Lateral Movement':      {'current': 65, 'target': 90, 'gaps': ['RDP lateral movement between subnets', 'WinRM abuse']},
    'Collection':            {'current': 55, 'target': 80, 'gaps': ['SharePoint bulk download', 'Email forwarding rules']},
    'Exfiltration':          {'current': 60, 'target': 90, 'gaps': ['DNS tunneling', 'Encrypted channel exfil', 'Cloud storage exfil']},
    'Impact':                {'current': 75, 'target': 95, 'gaps': ['Crypto-mining detection', 'Resource hijacking']},
}

print('=== MITRE ATT&CK Coverage Gap Analysis ===\n')
print(f'{"Tactic":<25} {"Current":<10} {"Target":<10} {"Gap":<8} Top Gaps')
print('─' * 100)
for tactic, data in MITRE_COVERAGE.items():
    gap = data['target'] - data['current']
    bar = '█' * (data['current'] // 5) + '░' * ((100 - data['current']) // 5)
    status = '⚠️' if gap > 20 else '✅' if gap <= 10 else '🔶'
    print(f'{status} {tactic:<23} {data["current"]:>3}%     {data["target"]:>3}%     {gap:>3}%   {data["gaps"][0]}')

avg_current = sum(d['current'] for d in MITRE_COVERAGE.values()) / len(MITRE_COVERAGE)
avg_target = sum(d['target'] for d in MITRE_COVERAGE.values()) / len(MITRE_COVERAGE)
print(f'\nOverall coverage: {avg_current:.0f}% current → {avg_target:.0f}% target')
print('Priority: Close gaps in Credential Access, Privilege Escalation, and Exfiltration first.')

In [ ]:
# ===================================================================
# ARCHITECTURE QUIZ: Test your understanding
# ===================================================================

QUIZ = [
    {
        'question': 'Woodgrove Bank needs to detect lateral movement from compromised on-prem AD accounts\n'
                    'to Azure resources. Which tool is BEST suited?',
        'options': {
            'A': 'Microsoft Defender for Endpoint',
            'B': 'Microsoft Defender for Identity',
            'C': 'Microsoft Sentinel with Entra ID + AD DS connectors',
            'D': 'Microsoft Defender for Cloud Apps',
        },
        'answer': 'C',
        'explanation': 'This requires correlating on-prem AD signals with Azure activity — that\'s cross-domain '
                       'correlation which is Sentinel\'s strength. Defender for Identity detects on-prem AD attacks '
                       'but doesn\'t natively correlate with Azure resource access. Sentinel can join both data sources '
                       'with KQL analytics rules.',
    },
    {
        'question': 'Which data tier should Woodgrove use for storing Azure Activity Logs in Sentinel\n'
                    'that are needed for compliance but rarely searched?',
        'options': {
            'A': 'Analytics (hot) tier',
            'B': 'Basic logs tier',
            'C': 'Archive tier',
            'D': 'Send to a separate Storage Account instead',
        },
        'answer': 'C',
        'explanation': 'Archive tier is designed for compliance data that must be retained but is rarely queried. '
                       'It\'s the cheapest tier in Sentinel. You can run search jobs against archived data when needed. '
                       'Basic logs are for data you occasionally search; Analytics is for active security monitoring.',
    },
    {
        'question': 'Woodgrove wants to automatically disrupt an in-progress ransomware attack.\n'
                    'Which capability provides this?',
        'options': {
            'A': 'Sentinel automation rules',
            'B': 'Sentinel playbooks (Logic Apps)',
            'C': 'Defender XDR automatic attack disruption',
            'D': 'Defender for Cloud security alerts',
        },
        'answer': 'C',
        'explanation': 'Automatic attack disruption is a unique Defender XDR capability that uses AI to identify '
                       'in-progress attacks and automatically contain them (e.g., isolating compromised devices, '
                       'disabling compromised accounts) without waiting for analyst action. Sentinel playbooks '
                       'require a trigger and workflow — they\'re not real-time attack disruption.',
    },
]

print('=== Architecture Design Quiz ===\n')
for i, q in enumerate(QUIZ, 1):
    print(f'Question {i}:')
    print(f'{q["question"]}\n')
    for key, option in q['options'].items():
        marker = '>>>' if key == q['answer'] else '   '
        print(f'  {marker} {key}. {option}')
    print(f'\n  Answer: {q["answer"]}')
    print(f'  Why: {q["explanation"]}')
    print()